In [ ]:
import sys
import subprocess

print("Installing dependencies (NVIDIA PhysicsNeMo, Scikit-Learn, Matplotlib, Pandas, NumPy)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "nvidia-physicsnemo", "scikit-learn", "matplotlib", "pandas", "numpy", "scipy"
])

import os
import glob
import time
import copy
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import physicsnemo
from physicsnemo.models.mlp.fully_connected import FullyConnected

BASE_PATH = os.getcwd()

DATASET_DIR = os.path.join(BASE_PATH, "dataset")
MODEL_DIR = os.path.join(BASE_PATH, "models")
FIG_DIR = os.path.join(BASE_PATH, "figures")
METRICS_DIR = os.path.join(BASE_PATH, "research_paper_metrics")

for d in [DATASET_DIR, MODEL_DIR, FIG_DIR, METRICS_DIR]:
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 80)
print("EXPERIMENT 1 RETRAINING PIPELINE INITIALIZED")
print("=" * 80)
print(f"Target Directory : {BASE_PATH}")
print(f"Execution Device : {device}")
print("=" * 80)


In [ ]:
all_csvs = glob.glob(os.path.join(DATASET_DIR, "*.csv")) + glob.glob(os.path.join(BASE_PATH, "*.csv")) + glob.glob(os.path.join(BASE_PATH, "**", "*.csv"), recursive=True)

all_csvs = list(dict.fromkeys(all_csvs))
ignore_kw = ["summary", "matrix", "history", "metrics", "eval", "outliers", "ablation", "paper", "system"]
candidate_csvs = [f for f in all_csvs if not any(kw in os.path.basename(f).lower() for kw in ignore_kw)]

DATASET_CSV = None
for csv_path in sorted(candidate_csvs, key=os.path.getmtime, reverse=True):
    try:
        temp_df = pd.read_csv(csv_path, nrows=5)
        if any(col.startswith("thickness_pt_") for col in temp_df.columns):
            DATASET_CSV = csv_path
            break
    except Exception:
        continue

if DATASET_CSV is None:
    raise FileNotFoundError(f"Could not locate raw spatial dataset CSV inside '{BASE_PATH}'.")

df_full = pd.read_csv(DATASET_CSV)
N_TOTAL = len(df_full)

FEATURE_COLS = ["temperature_C", "pulse_time_s", "num_cycles", "tma_mole_fraction"]
TARGET_COLS = [col for col in df_full.columns if col.startswith("thickness_pt_")]
N_SPATIAL_POINTS = len(TARGET_COLS)

np.random.seed(42)
torch.manual_seed(42)

all_indices = np.random.permutation(N_TOTAL)
N_TEST = min(2000, int(0.2 * N_TOTAL))
N_TRAIN = min(8000, N_TOTAL - N_TEST)

test_idx = all_indices[:N_TEST]
train_idx = all_indices[N_TEST:N_TEST + N_TRAIN]

df_test = df_full.iloc[test_idx].reset_index(drop=True)
df_train = df_full.iloc[train_idx].reset_index(drop=True)

X_train_raw = df_train[FEATURE_COLS].values.astype(np.float32)
Y_train_raw = df_train[TARGET_COLS].values.astype(np.float32)

X_mean, X_std = X_train_raw.mean(0), X_train_raw.std(0) + 1e-8
Y_mean, Y_std = Y_train_raw.mean(0), Y_train_raw.std(0) + 1e-8

X_test_raw = df_test[FEATURE_COLS].values.astype(np.float32)
Y_test_raw = df_test[TARGET_COLS].values.astype(np.float32)

X_train_norm = (X_train_raw - X_mean) / X_std
Y_train_norm = (Y_train_raw - Y_mean) / Y_std

X_test_norm = (X_test_raw - X_mean) / X_std

X_tr_tensor = torch.tensor(X_train_norm, device=device)
Y_tr_tensor = torch.tensor(Y_train_norm, device=device)
X_te_tensor = torch.tensor(X_test_norm, device=device)

print("=" * 80)
print("DATASET PARTITIONING COMPLETE")
print("=" * 80)
print(f"Total Dataset Size   : {N_TOTAL} samples")
print(f"Training Pool        : {len(df_train)} samples (Used for fitting & training)")
print(f"Holdout Test Set     : {len(df_test)} samples (Locked, unseen evaluation set)")
print(f"Spatial Coordinates  : {N_SPATIAL_POINTS} points")
print("=" * 80)


In [ ]:
model_name = "model_exp1_baseline_N8000"
ckpt_path = os.path.join(MODEL_DIR, f"{model_name}_checkpoint.pt")
final_path = os.path.join(MODEL_DIR, f"{model_name}_final.pt")

N_EPOCHS = 2500
PATIENCE = 300

surrogate = FullyConnected(
    in_features=len(FEATURE_COLS),
    out_features=N_SPATIAL_POINTS,
    num_layers=6,
    layer_size=128,
).to(device)

optimizer = torch.optim.Adam(surrogate.parameters(), lr=2e-3, weight_decay=5e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

start_epoch = 0
best_loss = float('inf')
epochs_no_improve = 0
best_state = None

if os.path.exists(ckpt_path):
    print(f"Resuming {model_name} from active checkpoint...")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    surrogate.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_loss = ckpt["best_loss"]
    best_state = ckpt["best_state"]

print("=" * 80)
print(f"STARTING MODEL 1 SURROGATE TRAINING ({len(df_train)} Training Samples)")
print("=" * 80)

t_start = time.perf_counter()

for epoch in range(start_epoch, N_EPOCHS):
    surrogate.train()
    optimizer.zero_grad()

    pred_norm = surrogate(X_tr_tensor)
    loss = torch.nn.MSELoss()(pred_norm, Y_tr_tensor)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(surrogate.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    loss_val = loss.item()

    if loss_val < best_loss:
        best_loss = loss_val
        epochs_no_improve = 0
        best_state = copy.deepcopy(surrogate.state_dict())
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 500 == 0 or epoch == N_EPOCHS - 1:
        print(f"  Epoch {epoch+1:4d}/{N_EPOCHS} | Training Loss: {loss_val:.6e}")

        torch.save({
            "epoch": epoch,
            "model_state_dict": surrogate.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_loss": best_loss,
            "best_state": best_state
        }, ckpt_path)

    if epochs_no_improve >= PATIENCE:
        print(f"  Early stopping triggered at Epoch {epoch+1}.")
        break

t_end = time.perf_counter()
train_time_sec = round(t_end - t_start, 2)

if best_state is not None:
    surrogate.load_state_dict(best_state)

torch.save({
    "model_name": model_name,
    "training_samples": len(df_train),
    "model_state_dict": surrogate.state_dict(),
    "X_mean": X_mean, "X_std": X_std,
    "Y_mean": Y_mean, "Y_std": Y_std
}, final_path)

print("\n" + "=" * 80)
print(f"MODEL 1 TRAINING COMPLETED IN {train_time_sec} SECONDS")
print(f"Serialized Model Saved To: {final_path}")
print("=" * 80)


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

surrogate.eval()

with torch.no_grad():
    Y_test_pred_norm = surrogate(X_te_tensor).cpu().numpy()
Y_test_pred_nm = Y_test_pred_norm * Y_std + Y_mean

abs_err = np.abs(Y_test_raw - Y_test_pred_nm)
sq_err = (Y_test_raw - Y_test_pred_nm) ** 2

global_r2 = float(r2_score(Y_test_raw.flatten(), Y_test_pred_nm.flatten()))
global_mae = float(abs_err.mean())
global_rmse = float(np.sqrt(sq_err.mean()))
global_max_err = float(abs_err.max())

p50 = float(np.percentile(abs_err, 50))
p95 = float(np.percentile(abs_err, 95))
p99 = float(np.percentile(abs_err, 99))

boundary_w = max(1, int(0.05 * N_SPATIAL_POINTS))
fin_peak_pts = list(range(0, boundary_w)) + list(range(N_SPATIAL_POINTS - boundary_w, N_SPATIAL_POINTS))
sidewall_pts = list(range(boundary_w, int(0.40 * N_SPATIAL_POINTS))) + list(range(int(0.60 * N_SPATIAL_POINTS), N_SPATIAL_POINTS - boundary_w))
trench_bottom_pts = list(range(int(0.40 * N_SPATIAL_POINTS), int(0.60 * N_SPATIAL_POINTS)))
corner_pts = list(range(int(0.38 * N_SPATIAL_POINTS), int(0.42 * N_SPATIAL_POINTS))) + list(range(int(0.58 * N_SPATIAL_POINTS), int(0.62 * N_SPATIAL_POINTS)))

mae_fin = float(abs_err[:, fin_peak_pts].mean())
mae_side = float(abs_err[:, sidewall_pts].mean())
mae_bot = float(abs_err[:, trench_bottom_pts].mean())
mae_corner = float(abs_err[:, corner_pts].mean())

print("=" * 80)
print("EXACT METRICS TO PASTE INTO SLIDES 1 - 4 (2,000 HOLDOUT TEST SET)")
print("=" * 80)
print(f"  * Global Spatial R² Score             : {global_r2:.8f}")
print(f"  * Global Mean Absolute Error (MAE)    : {global_mae:.6e} nm ({global_mae * 10.0:.4f} Å)")
print(f"  * Global Root Mean Squared Error(RMSE): {global_rmse:.6e} nm")
print(f"  * Maximum Pointwise Error Envelope    : {global_max_err:.6e} nm")
print(f"  * 50th Percentile (Median Error)      : {p50:.6e} nm")
print(f"  * 95th Percentile Tail Threshold      : {p95:.6e} nm")
print(f"  * 99th Percentile Extreme Outlier     : {p99:.6e} nm")
print(f"  * Fin Peak MAE (Points 0-5, 94-99)    : {mae_fin:.6e} nm")
print(f"  * Sidewall MAE (Points 6-39, 61-93)   : {mae_side:.6e} nm")
print(f"  * Trench Bottom MAE (Points 40-60)    : {mae_bot:.6e} nm")
print(f"  * Corner Discontinuity MAE            : {mae_corner:.6e} nm")
print("=" * 80)


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes1[0].imshow(Y_test_raw, aspect='auto', cmap='viridis', origin='upper')
axes1[0].set_title('ViennaPS Ground Truth Thickness (nm)', fontweight='bold', fontsize=12)
axes1[0].set_ylabel('Process Sample Index', fontweight='bold')
axes1[0].set_xlabel('Spatial Coordinate Index (0 = Left Fin, 50 = Bottom, 99 = Right Fin)', fontweight='bold')
fig1.colorbar(im0, ax=axes1[0], fraction=0.046, pad=0.04)

im1 = axes1[1].imshow(Y_test_pred_nm, aspect='auto', cmap='viridis', origin='upper')
axes1[1].set_title('NVIDIA PhysicsNeMo Predicted Thickness (nm)', fontweight='bold', fontsize=12)
axes1[1].set_xlabel('Spatial Coordinate Index', fontweight='bold')
fig1.colorbar(im1, ax=axes1[1], fraction=0.046, pad=0.04)

im2 = axes1[2].imshow(abs_err, aspect='auto', cmap='inferno', origin='upper')
axes1[2].set_title('Absolute Spatial Error |Y_true - Y_pred| (nm)', fontweight='bold', fontsize=12)
axes1[2].set_xlabel('Spatial Coordinate Index', fontweight='bold')
fig1.colorbar(im2, ax=axes1[2], fraction=0.046, pad=0.04)

plt.tight_layout()
fig1_path = os.path.join(FIG_DIR, "spatial_heatmaps_full_n2000.png")
plt.savefig(fig1_path, dpi=250, bbox_inches='tight')
plt.show()
plt.close(fig1)

fig2, axes2 = plt.subplots(1, 2, figsize=(14, 4.5))

axes2[0].hist(abs_err.flatten(), bins=100, color='#6baed6', edgecolor='#3182bd', alpha=0.85, log=True)
axes2[0].axvline(p95, color='brown', linestyle='--', linewidth=1.5, label=f'95th Percentile ({p95:.6f} nm)')
axes2[0].axvline(p99, color='maroon', linestyle=':', linewidth=2, label=f'99th Percentile ({p99:.6f} nm)')

axes2[0].set_title('Spatial Error Probability Density & Tail', fontweight='bold', fontsize=12)
axes2[0].set_xlabel('Absolute Pointwise Spatial Error (nm)', fontweight='bold')
axes2[0].set_ylabel('Probability Density (Log Scale)', fontweight='bold')
axes2[0].grid(True, which='both', linestyle='-', alpha=0.25)
axes2[0].legend(loc='upper right', framealpha=0.9)

sorted_err = np.sort(abs_err.flatten())
cdf_y = np.linspace(0, 1, len(sorted_err))

axes2[1].plot(sorted_err, cdf_y, color='#2ca02c', linewidth=2.0)
axes2[1].axhline(0.95, color='red', linestyle='--', alpha=0.7, label='95% Coverage')
axes2[1].axvline(p95, color='red', linestyle='--', alpha=0.7)

axes2[1].set_title('Cumulative Error Distribution (CDF)', fontweight='bold', fontsize=12)
axes2[1].set_xlabel('Absolute Spatial Error (nm)', fontweight='bold')
axes2[1].set_ylabel('Cumulative Probability', fontweight='bold')
axes2[1].grid(True, linestyle='-', alpha=0.25)
axes2[1].legend(loc='lower right', framealpha=0.9)

plt.tight_layout()
fig2_path = os.path.join(FIG_DIR, "error_tail_distribution_n2000.png")
plt.savefig(fig2_path, dpi=250, bbox_inches='tight')
plt.show()
plt.close(fig2)

print("=" * 80)
print("EXACT MATCHING FIGURES GENERATED & DISPLAYED IN CONSOLE")
print("=" * 80)
print(f"Slide 1 Visual Saved To : {fig1_path}")
print(f"Slide 2 Visual Saved To : {fig2_path}")
print("=" * 80)
